In [1]:
import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
import torch.nn.functional as F

import kagglehub
from kagglehub import KaggleDatasetAdapter

from transformers import get_cosine_schedule_with_warmup

from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter

from pathlib import Path
import shutil

In [2]:
file_path = "likes.parquet"

data = (
    kagglehub.dataset_load(
        KaggleDatasetAdapter.POLARS,
        "thekabeton/ysda-recsys-2026-yambda-dataset/versions/3",
        file_path,
    ).collect()
    # .sample(1000000)
)

In [3]:
lf = pl.scan_parquet("./generated/yambda_data/flat/5b/dislikes.parquet")
dislikes = lf.select(
    ["uid", "timestamp", "item_id", "is_organic"]
).collect()  # .sample(1000000)

In [4]:
val_size = 60 * 60 * 24
gap_size = 60 * 30
# max_timestamp = data.select(pl.col("timestamp").max()).item()

train_data = data.filter(
    pl.col("timestamp") < data["timestamp"].max() - val_size - gap_size
)

dislikes = dislikes.filter(
    pl.col("timestamp") < data["timestamp"].max() - val_size - gap_size
)

val_data = data.filter(
    pl.col("timestamp") >= data["timestamp"].max() - val_size
)
val_data = val_data.join(
    train_data.select("uid").unique(),
    on="uid",
    how="semi",
)

SPECIAL_TOKENS = ["PAD", "BOS", "UNK"]
item2ind = dict(
    train_data.select("item_id")
    .unique()
    .sort("item_id")
    .with_row_index("ind", offset=len(SPECIAL_TOKENS))
    .select(["item_id", "ind"])
    .iter_rows()
)

train_data = train_data.with_columns(
    pl.col("item_id")
    .replace_strict(item2ind, default=None)
    .cast(pl.UInt32)
    .alias("item_ind")
)

val_data = val_data.with_columns(
    pl.col("item_id")
    .replace_strict(item2ind, default=2)
    .cast(pl.UInt32)
    .alias("item_ind")
)

dislikes = dislikes.with_columns(
    pl.col("item_id")
    .replace_strict(item2ind, default=2)
    .cast(pl.UInt32)
    .alias("item_ind")
)

In [ ]:
from dataset import TrainDataset, ValDataset
from sasrec import SASRec

In [7]:
def rowwise_intersection_size_set(
    A: torch.Tensor, B: torch.Tensor, pad: int | None = None
) -> torch.Tensor:
    """
    A: [B, T]
    B: [B, K]
    Returns: [B]  (count of distinct values common to both rows)
    """
    if pad is not None:
        A = A.masked_fill(A == pad, -1)
        B = B.masked_fill(B == pad, -2)

    # [B, T, K] equality; then "for each A element, does it appear in B row?"
    hits = A.unsqueeze(2) == B.unsqueeze(1)  # bool [B, T, K]
    in_B = hits.any(dim=2)  # bool [B, T]

    # remove duplicates in A by counting each distinct A value at most once
    # sort A and only keep first occurrence positions
    A_sorted, _ = torch.sort(A, dim=1)
    is_first = torch.ones_like(A_sorted, dtype=torch.bool)
    is_first[:, 1:] = A_sorted[:, 1:] != A_sorted[:, :-1]

    # align in_B to sorted order
    # easiest: compute in_B for sorted A vs B instead of original A
    hits_sorted = A_sorted.unsqueeze(2) == B.unsqueeze(1)
    in_B_sorted = hits_sorted.any(dim=2)

    out = (in_B_sorted & is_first).sum(dim=1)  # [B]

    return out

In [8]:
def loss_fn(input, target, negatives, model):
    hidden = model.get_hidden(input)

    items = torch.cat([target.unsqueeze(2), negatives], dim=2)
    item_emb = model.item_emb(items)

    scores = (item_emb @ hidden.unsqueeze(3))[:, :, :, 0]
    scores = scores.masked_fill(items == 0, float("-inf"))

    log_probs = F.log_softmax(scores, dim=2)
    pos_logp = log_probs[..., 0]

    mask = target != 0
    loss = -(pos_logp[mask]).mean()

    return loss


def save_checkpoint(
    model, optimizer, scheduler, epoch, model_name, metric=None
):
    ckpt_dir = Path("checkpoints") / model_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    ckpt = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "metric": float(metric) if metric is not None else None,
    }
    path = ckpt_dir / "last.pt"
    torch.save(ckpt, path)
    return path


def train(
    model,
    optimizer,
    scheduler,
    model_name,
    train_dataloader,
    val_dataloder,
    EPOCHS,
    continue_training=True,
):
    start_epoch = 0
    if continue_training:
        ckpt_dir = Path("checkpoints") / model_name
        path = ckpt_dir / "last.pt"

        ckpt = torch.load(path)

        start_epoch = ckpt["epoch"] + 1
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    else:
        shutil.rmtree(Path("runs") / model_name, ignore_errors=True)
    writer = SummaryWriter(log_dir=f"runs/{model_name}")

    best_recall = 0
    for epoch in range(start_epoch, EPOCHS):
        train_epoch(
            model, optimizer, scheduler, train_dataloader, writer, epoch
        )
        save_checkpoint(model, optimizer, scheduler, epoch, model_name)

        recall = val_model(model, val_dataloder, writer, epoch)
        print(f"Recal at epoch {epoch} = {recall}")

        if recall > best_recall:
            best_recall = recall
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "metric": best_recall,
                },
                Path("checkpoints") / f"{model_name}_best.pt",
            )

    writer.close()


def train_epoch(model, optimizer, scheduler, dataloader, writer, epoch):
    device = next(model.parameters()).device
    model.train()

    for step, batch in enumerate(tqdm(dataloader)):
        input = batch["input"].to(device, non_blocking=True)
        target = batch["target"].to(device, non_blocking=True)
        negatives = batch["negatives"].to(device, non_blocking=True)

        loss = loss_fn(input, target, negatives, model)
        writer.add_scalar("train/loss", loss, step + epoch * len(dataloader))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()

        writer.add_scalar(
            "train/lr",
            scheduler.get_last_lr()[0],
            step + epoch * len(dataloader),
        )


@torch.no_grad()
def val_model(model, dataloader, writer, epoch):
    device = next(model.parameters()).device
    model.eval()

    topk = 100
    recall, recall_n = 0, 0
    for batch in tqdm(dataloader):
        history = batch["history"].to(device, non_blocking=True)
        target = batch["target"].to(device, non_blocking=True)

        ranking = model.top_k(history, topk)

        lenght = (target != 0).sum(1)
        recall += (
            rowwise_intersection_size_set(target, ranking)
            / torch.minimum(lenght, torch.tensor(topk, device=device))
        ).sum()
        recall_n += target.size(0)

    writer.add_scalar("val/recall@100", recall / recall_n, epoch)
    return recall / recall_n

In [9]:
model = SASRec()
model.to("cuda")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

NEGATIVE_COUNT = 100
train_dataset = TrainDataset(
    train_data, dislikes, negatives_count=NEGATIVE_COUNT
)
val_dataset = ValDataset(train_data, val_data)


BATCH_SIZE = 256
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=8,
    prefetch_factor=4,
    pin_memory=True,
    persistent_workers=True,
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    prefetch_factor=2,
    pin_memory=True,
    persistent_workers=True,
)

EPOCHS = 10
total_steps = EPOCHS * len(train_dataloader)
warmup_steps = int(0.05 * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

train(
    model,
    optimizer,
    scheduler,
    "baseline",
    train_dataloader,
    val_dataloader,
    EPOCHS,
    continue_training=False,
)

/home/sonofr/python_venvs/.venv/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


18170


  0%|          | 0/3228 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

Recal at epoch 0 = 0.04969872161746025


...

  0%|          | 0/500 [00:00<?, ?it/s]

Recal at epoch 8 = 0.07818371057510376


  0%|          | 0/3228 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

Recal at epoch 9 = 0.07795841991901398
